# Section 1 — Imports and Logging

Imports copied from `MANGODisplay_OSN.py`. Defines helper functions used in later sections.

In [ ]:
from datetime import datetime, timedelta
import os
from glob import glob
from matplotlib import ticker, gridspec, dates
from cartopy import crs, feature
import gc
import numpy as np
import airglow.prepare_agimages as prepare_agimages
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import calendar
import pytz
import airglow.plotmangodasi as pmd
import airglow.fpiinfo as fpiinfo
from os.path import exists
import shutil
import matplotlib.pyplot as plt
import argparse
import airglow.BoltwoodSensor as BoltwoodSensor
import subprocess
import airglow.asiinfo as asiinfo
import ephem
import airglow.MANGO_L2 as MANGO_L2
import pandas as pd
import xarray as xr
from matplotlib.lines import Line2D
import logging
from pathlib import Path
from airglow.cloud_storage import CloudStorage, Configuration
import warnings


def setup_logging():
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[logging.StreamHandler()]
    )
    return logging.getLogger(__name__)


def read_FPI(fname, reference='zenith'):
    npzfile = np.load(fname, allow_pickle='False', encoding='latin1')
    FPI_Results = npzfile['FPI_Results']
    FPI_Results = FPI_Results.reshape(-1)[0]
    del npzfile.f
    npzfile.close()

    FPI_ut = {}
    FPI_wind = {}
    FPI_error = {}
    FPI_cloud = {}
    FPI_wq = {}

    FPI_ut['North'], FPI_wind['North'], FPI_error['North'], FPI_cloud['North'], FPI_wq['North'] = Process_FPI(FPI_Results, 'North', reference=reference)
    FPI_ut['East'],  FPI_wind['East'],  FPI_error['East'],  FPI_cloud['East'],  FPI_wq['East']  = Process_FPI(FPI_Results, 'East',  reference=reference)
    FPI_ut['South'], FPI_wind['South'], FPI_error['South'], FPI_cloud['South'], FPI_wq['South'] = Process_FPI(FPI_Results, 'South', reference=reference)
    FPI_ut['West'],  FPI_wind['West'],  FPI_error['West'],  FPI_cloud['West'],  FPI_wq['West']  = Process_FPI(FPI_Results, 'West',  reference=reference)
    FPI_ut['Zenith'],FPI_wind['Zenith'],FPI_error['Zenith'],FPI_cloud['Zenith'],FPI_wq['Zenith']= Process_FPI(FPI_Results, 'Zenith',reference=reference)

    return FPI_ut, FPI_wind, FPI_error, FPI_cloud, FPI_wq


def Process_FPI(FPI_Results, desired_dir, reference='zenith'):
    import airglow.FPI as FPI
    from scipy import interpolate

    (ref_Dop, e_ref_Dop) = FPI.DopplerReference(FPI_Results, reference=reference)

    ind = FPI.all_indices('Zenith', FPI_Results['direction'])
    w = (FPI_Results['LOSwind'][ind] - ref_Dop[ind])
    sigma_w = FPI_Results['sigma_LOSwind'][ind]
    dt = []
    for x in FPI_Results['sky_times'][ind]:
        diff = (x - FPI_Results['sky_times'][0])
        dt.append(diff.seconds + diff.days * 86400.)
    dt = np.array(dt)

    ind = abs(w) < 200.

    if sum(ind) <= 1:
        ind = np.array([True for i in range(len(w))])

    if len(ind) == 0:
        raise Exception('No Zenith look directions')

    w2      = interpolate.interp1d(dt[ind], w[ind],       bounds_error=False, fill_value=0.0)
    sigma_w2= interpolate.interp1d(dt[ind], sigma_w[ind], bounds_error=False, fill_value=0.0)
    dt = []

    for x in FPI_Results['sky_times']:
        diff = (x - FPI_Results['sky_times'][0])
        dt.append(diff.seconds + diff.days * 86400.)
    w       = w2(dt)
    sigma_w = sigma_w2(dt)

    ind = FPI.all_indices(desired_dir, FPI_Results['direction'])

    if desired_dir == 'Zenith':
        Doppler_Wind  = (FPI_Results['LOSwind'][ind] - ref_Dop[ind])
        Doppler_Error = np.sqrt(FPI_Results['sigma_LOSwind'][ind]**2)
    else:
        Doppler_Wind  = (FPI_Results['LOSwind'][ind] - ref_Dop[ind] - w[ind] * np.cos(FPI_Results['ze'][ind] * np.pi / 180.)) / np.sin(FPI_Results['ze'][ind] * np.pi / 180.)
        Doppler_Error = np.sqrt(FPI_Results['sigma_LOSwind'][ind]**2 + sigma_w[ind]**2)
    if desired_dir == 'South' or desired_dir == 'West':
        Doppler_Wind = -Doppler_Wind

    return FPI_Results['sky_times'][ind], Doppler_Wind, Doppler_Error, FPI_Results['Clouds']['mean'][ind], FPI_Results['wind_quality_flag'][ind]


def toTimestamp(d):
    return calendar.timegm(d.timetuple())


def get_parameters(sky_line_tag, analysis_parameters):
    params = {}
    if sky_line_tag == 'XG':
        params['figsize']       = (8, 4)
        params['width_ratios']  = [2., 2]
        params['extent']        = [-122, -95, 25, 55]
        params['ymin']          = -200
        params['ymax']          = 200
        params['title']         = 'Green Line Winds'
        params['scale']         = 25.
        params['xoff'], params['yoff'] = 0.06, 0.05
        params['cloc']          = [0.7, -0.0, 0.4, 0.02]
        if np.isnan(analysis_parameters['Thi']):
            params['cmap'] = 'viridis'
        else:
            params['cmap'] = 'Greens_r'
    elif sky_line_tag == 'XR':
        params['figsize']       = (9, 4)
        params['width_ratios']  = [2.2, 2]
        params['extent']        = [-125, -75, 25, 55]
        params['ymin']          = -300
        params['ymax']          = 300
        params['title']         = 'Red Line Winds'
        params['scale']         = 100.
        params['xoff'], params['yoff'] = 0.06, 0.05
        params['cloc']          = [0.55, -0.02, 0.5, 0.02]
        if np.isnan(analysis_parameters['Thi']):
            params['cmap'] = 'inferno'
        else:
            params['cmap'] = 'Reds'
    return params


def download_fpi_data(cloud_storage, config, site, date, sky_line_tag, fpi_dir):
    logger = logging.getLogger(__name__)
    created_files = []

    instr_name   = fpiinfo.get_instr_at(site, date)[0]
    fpi_datestr  = date.strftime('%Y%m%d')
    fpi_filename = f"{instr_name}_{site}_{fpi_datestr}_{sky_line_tag.lower()}.npz"
    print(f"download_fpi: {fpi_filename}")

    cloud_key      = f"{config.aws_results_prefix}{date.strftime('%Y')}/{fpi_filename}"
    local_fpi_path = os.path.join(fpi_dir, fpi_filename)

    os.makedirs(os.path.dirname(local_fpi_path), exist_ok=True)

    if cloud_storage.download_file(cloud_key, local_fpi_path):
        logger.info(f"Downloaded FPI file: {fpi_filename}")
        created_files.append(local_fpi_path)
    else:
        if sky_line_tag == 'XR':
            fpi_filename   = f"{instr_name}_{site}_{fpi_datestr}.npz"
            cloud_key      = f"{config.aws_results_prefix}{fpi_filename}"
            local_fpi_path = os.path.join(fpi_dir, fpi_filename)
            if cloud_storage.download_file(cloud_key, local_fpi_path):
                logger.info(f"Downloaded FPI file (without tag): {fpi_filename}")
                created_files.append(local_fpi_path)
            else:
                logger.warning(f"Failed to download FPI file for {site} on {fpi_datestr}")
        else:
            logger.warning(f"Failed to download FPI file for {site} on {fpi_datestr}")

    return created_files, local_fpi_path


logger = setup_logging()


# Section 2 — Configuration

Edit the values in this cell before running Section 3.

In [ ]:
# ── Date ──────────────────────────────────────────────────────────────────────
year = 2024
doy  = 100          # Day of year

# ── Emission tag ──────────────────────────────────────────────────────────────
# 'XG' -> greenline (557.7 nm)   'XR' -> redline (630.0 nm)
sky_line_tag = 'XG'

# ── Working directories (local scratch space) ─────────────────────────────────
ASI_directory    = '/tmp/mango/asi/'
FPI_directory    = '/tmp/mango/fpi/'
output_directory = '/tmp/mango/frames/'

# ── OSN credentials ───────────────────────────────────────────────────────────
env_file = '.env'   # path to .env file used by Configuration / CloudStorage

# ── Download flags ────────────────────────────────────────────────────────────
download_asi_data      = True    # set False if ASI HDF5 files are already on disk
download_fpi_data_flag = True    # set False if FPI .npz files are already on disk

# ── Derived parameters (do not edit below this line) ─────────────────────────
analysis_date = datetime(year, 1, 1) + timedelta(days=doy - 1)
fpi_date      = analysis_date + timedelta(days=-1)   # FPI files named for previous calendar day

if sky_line_tag == 'XG':
    sites_asi = ['cvo', 'low', 'blo', 'cfs', 'mro', 'bdr', 'new']
    sites_fpi = ['cvo', 'low', 'blo']
    emission  = 'green'
    Tlo, Thi  = 2, 20
elif sky_line_tag == 'XR':
    sites_asi = ['cfs', 'cvo', 'eio', 'mdk', 'mto', 'par']
    sites_fpi = ['cvo', 'low', 'blo', 'uao']
    emission  = 'red'
    Tlo, Thi  = 8, 30

analysis_parameters = {
    'date':        analysis_date,
    'ntaps':       13,
    'emission':    emission,
    'el_cutoff':   20.,
    'Tlo':         Tlo,
    'Thi':         Thi,
    'sites_asi':   sites_asi,
    'sites_fpi':   sites_fpi,
    'download_data': download_asi_data,
}

system_parameters = {
    'ASI_directory':    ASI_directory,
    'FPI_directory':    FPI_directory,
    'output_directory': output_directory,
}

for d in system_parameters.values():
    os.makedirs(d, exist_ok=True)

config        = Configuration(env_file)
cloud_storage = CloudStorage(config)
params        = get_parameters(sky_line_tag, analysis_parameters)

my_colors = {'cvo': '#1f77b4', 'blo': '#ff7f0e', 'low': '#9467bd', 'uao': '#8c564b'}
max_gap     = 120.    # minutes — max gap in FPI wind series before line is broken
max_fpi_gap = 30.     # minutes — max gap before FPI quiver arrow is hidden

logger.info(f"Configured for {analysis_date.strftime('%Y-%m-%d')} ({sky_line_tag})")


# Section 3 — Data Loading

Run once per night. Loads all ASI imagery and FPI winds into memory.

In [ ]:
# Run once per night — loads and preprocesses all ASI and FPI data

# ── 3a: Download and load ASI data ───────────────────────────────────────────
if download_asi_data:
    for site in analysis_parameters['sites_asi']:
        try:
            if sky_line_tag == 'XG':
                cmd = '/usr/bin/wget -r -nH --cut-dirs=8 --no-parent -P %s/%s/%s https://data.mangonetwork.org/data/transport/mango/archive/%s/greenline/level1/%04d/%s/' % \
                    (system_parameters['ASI_directory'], site, analysis_parameters['date'].year, site, analysis_parameters['date'].year, analysis_parameters['date'].strftime('%j'))
            elif sky_line_tag == 'XR':
                cmd = '/usr/bin/wget -r -nH --cut-dirs=8 --no-parent -P %s/%s/%s https://data.mangonetwork.org/data/transport/mango/archive/%s/redline/level1/%04d/%s/' % \
                    (system_parameters['ASI_directory'], site, analysis_parameters['date'].year, site, analysis_parameters['date'].year, analysis_parameters['date'].strftime('%j'))
            logger.info(f"Downloading ASI data for {site}: {cmd}")
            MANGO_L2.runcmd(cmd)
        except Exception as e:
            logger.error(f"Failure to download {site} on {analysis_parameters['date']}: {str(e)}")

ds = {}
for site in analysis_parameters['sites_asi']:
    files = glob('%s/%s/%s/*%s*.hdf5' % (
        system_parameters['ASI_directory'], site,
        analysis_parameters['date'].strftime('%Y/%j'),
        analysis_parameters['emission']))
    if len(files) > 0:
        try:
            ds[site] = MANGO_L2.load_hdf5_to_xarray(files[0])
        except Exception as e:
            logger.warning(f"Failed to load ASI data for {site}: {e}")
    else:
        logger.warning(f"No ASI HDF5 files found for {site}")


In [ ]:
# ── 3b: Apply temporal filtering to ASI data ─────────────────────────────────
for site in list(ds.keys()):
    logger.info(f"Processing images for site: {site}")
    IM3D  = ds[site].ImageData.values
    IM3Dt = np.transpose(IM3D, (1, 2, 0))
    times = pd.to_datetime(ds[site].time.values)

    if np.isnan(analysis_parameters['Tlo']) and np.isnan(analysis_parameters['Thi']):
        IM3Dfilt = IM3Dt
    else:
        b        = prepare_agimages.initialize_airglow_filter(
                       analysis_parameters['ntaps'],
                       analysis_parameters['Tlo'],
                       analysis_parameters['Thi'], times)
        IM3Dfilt = prepare_agimages.filter_airglow(IM3Dt, b, analysis_parameters['ntaps'])

    filt_array = xr.DataArray(np.transpose(IM3Dfilt, (2, 0, 1)), dims=('time', 'north', 'east'))
    ds[site]['FilteredImageData'] = filt_array

    analysis_parameters['site'] = site
    results_df = MANGO_L2.load_cloud_and_moon(analysis_parameters, times)

    cloud_temperature = xr.DataArray(results_df['Cloud Temperature'], dims=('time'), coords={'time': times})
    moon_angle        = xr.DataArray(results_df['Moon Altitude'],     dims=('time'), coords={'time': times})

    ds[site]['cloud_temperature'] = cloud_temperature
    ds[site]['moon_angle']        = moon_angle
    ds[site]['cloud_temperature'].attrs['units'] = 'Celsius'
    ds[site]['moon_angle'].attrs['units']        = 'Degrees'


In [ ]:
# ── 3c: Download and load FPI data ───────────────────────────────────────────
fpi_dir = os.path.join(system_parameters['FPI_directory'])
os.makedirs(fpi_dir, exist_ok=True)

if download_fpi_data_flag:
    for fpi_site in analysis_parameters['sites_fpi']:
        files, _ = download_fpi_data(cloud_storage, config, fpi_site, fpi_date, sky_line_tag.lower(), fpi_dir)
        print(files)

FPI_ut     = {}
FPI_wind   = {}
FPI_error  = {}
FPI_cloud  = {}
FPI_wq     = {}
FPI_tt     = {}
FPI_walpha = {}

for fpi in analysis_parameters['sites_fpi']:
    fname = os.path.join(fpi_dir, f"{fpiinfo.get_instr_at(fpi, fpi_date)[0]}_{fpi}_{fpi_date.strftime('%Y%m%d')}_{sky_line_tag.lower()}.npz")
    print(fname)
    if (not exists(fname)) and (sky_line_tag == 'XR'):
        # Older redline files may not have the tag suffix
        fname = os.path.join(fpi_dir, f"{fpiinfo.get_instr_at(fpi, fpi_date)[0]}_{fpi}_{fpi_date.strftime('%Y%m%d')}.npz")

    if exists(fname):
        FPI_ut[fpi], FPI_wind[fpi], FPI_error[fpi], FPI_cloud[fpi], FPI_wq[fpi] = read_FPI(fname)
        FPI_tt[fpi]     = {}
        FPI_walpha[fpi] = {}
        for d in FPI_ut[fpi].keys():
            FPI_tt[fpi][d] = np.array([toTimestamp(d.astimezone(pytz.utc)) for d in FPI_ut[fpi][d]])
            FPI_walpha[fpi][d] = FPI_wq[fpi][d].copy()
            FPI_walpha[fpi][d][FPI_walpha[fpi][d] == 2] = 0.3
            FPI_walpha[fpi][d][FPI_walpha[fpi][d] == 1] = 0.7
            FPI_walpha[fpi][d][FPI_walpha[fpi][d] == 0] = 1.0
    else:
        logger.warning(f"FPI file not found for {fpi}, skipping")


In [ ]:
# ── 3d: Compute unique times list ─────────────────────────────────────────────
all_times = []
for k in ds.keys():
    if hasattr(ds[k], 'time') and ds[k].time is not None:
        all_times += list(ds[k].time.values)

if not all_times:
    raise RuntimeError("No time data found in ASI datasets — check downloads.")

all_times.sort()

all_fpi_times = []
for k in FPI_ut.keys():
    for d in FPI_ut[k].keys():
        all_fpi_times += list(FPI_ut[k][d])
all_fpi_times.sort()

all_unique_times = np.sort(np.unique(all_times))
unique_times = [all_unique_times[0]]
for i in range(1, len(all_unique_times)):
    if all_unique_times[i] - all_unique_times[i-1] > np.timedelta64(10, 's'):
        unique_times.append(all_unique_times[i])

# Pre-compute a single fixed colormap range across all sites and all frames
_clim_samples = []
for _site in ds.keys():
    if 'FilteredImageData' not in ds[_site]:
        continue
    _fid  = ds[_site]['FilteredImageData']
    _mask = (ds[_site]['Elevation'] > analysis_parameters['el_cutoff']).values
    _n    = len(ds[_site].time)
    _step = max(1, _n // 30)
    for _ti in range(0, _n, _step):
        _frame = _fid.isel(time=_ti).values
        _valid = _frame[_mask]
        _clim_samples.append(_valid[np.isfinite(_valid)])
if _clim_samples:
    _all_data = np.concatenate(_clim_samples)
    vmin = float(np.nanpercentile(_all_data, 2))
    vmax = float(np.nanpercentile(_all_data, 98))
else:
    vmin, vmax = None, None

logger.info(f"Loaded {len(ds)} ASI sites: {list(ds.keys())}")
logger.info(f"Loaded {len(FPI_ut)} FPI sites: {list(FPI_ut.keys())}")
logger.info(f"Found {len(unique_times)} unique time steps")
logger.info(f"Time range: {pd.to_datetime(str(unique_times[0]))} -> {pd.to_datetime(str(unique_times[-1]))}")
logger.info(f"Colormap limits: vmin={vmin}, vmax={vmax}")


# Section 4 — Frame Generation

Edit `target_time_str` and re-run this cell to render any frame from the loaded night.

In [ ]:
# ── Target time ───────────────────────────────────────────────────────────────
# Enter any UTC time string within the night, e.g. '2024-04-09 08:30:00'
# Or use: target_time = unique_times[0]  to get the first available frame
# Or use: target_time = unique_times[N]  to index directly into the frame list
target_time_str = '2024-04-09 08:30:00'

# Convert: accept either a string or a numpy datetime64 directly
if isinstance(target_time_str, str):
    target_time = np.datetime64(target_time_str)
else:
    target_time = target_time_str

# ── Validate target time is within loaded range ────────────────────────────────
if target_time < unique_times[0] or target_time > unique_times[-1]:
    raise ValueError(
        f"target_time {target_time} is outside the available range "
        f"{unique_times[0]} -> {unique_times[-1]}"
    )

# ── Build figure ──────────────────────────────────────────────────────────────
fig  = plt.figure(figsize=params['figsize'])
spec = gridspec.GridSpec(ncols=2, nrows=2, figure=fig,
                         left=0.04, right=0.94, bottom=0.1, top=0.90,
                         wspace=0.05, hspace=0.25,
                         width_ratios=params['width_ratios'])
legend_elements = []

axes00 = fig.add_subplot(spec[:, 0], projection=crs.Orthographic(np.nanmean(params['extent'][:2]), np.nanmean(params['extent'][2:])))
axes00.add_feature(feature.COASTLINE)
axes00.add_feature(feature.STATES, alpha=0.2, zorder=1)

pc = None
for k in ds.keys():
    if hasattr(ds[k], 'time') and ds[k].time is not None:
        nearest_time    = ds[k].sel(time=target_time, method='nearest')
        time_difference = abs(pd.to_timedelta(nearest_time.time.values - target_time).total_seconds())

        if time_difference < 10:
            data      = ds[k].sel(time=target_time, method='nearest')
            latitude  = data.Latitude
            longitude = data.Longitude
            mask      = data.Elevation > analysis_parameters['el_cutoff']
            pc        = axes00.pcolormesh(longitude, latitude, data.FilteredImageData.where(mask),
                                          transform=crs.PlateCarree(), cmap=params['cmap'],
                                          vmin=vmin, vmax=vmax)

axes00.set_title('%s UT' % pd.to_datetime(str(target_time)))
axes00.set_extent(params['extent'], crs=crs.PlateCarree())
axes00.set_facecolor('lightgray')

if pc is not None:
    pos  = axes00.get_position()
    cax  = fig.add_axes([pos.x0 + params['cloc'][0] * pos.width, pos.y0 + params['cloc'][1],
                         params['cloc'][2] * pos.width - 0.05, params['cloc'][3]])
    cbar = fig.colorbar(pc, cax=cax, orientation='horizontal')
    cbar.set_label('Intensity [units]', fontsize=8)
    cbar.set_ticks([])
    cbar.set_ticklabels([])
    cbar.ax.tick_params(labelsize=8)
else:
    print("Warning: no ASI sites had data within 10 s of target_time — colorbar skipped.")

axes01 = fig.add_subplot(spec[0, 1])
axes02 = fig.add_subplot(spec[1, 1])

for fpi in FPI_ut.keys():
    for i in range(len(FPI_ut[fpi]['North']) - 1):
        if (FPI_ut[fpi]['North'][i + 1] - FPI_ut[fpi]['North'][i]).total_seconds() < (max_gap * 60.):
            axes01.plot(FPI_ut[fpi]['North'][i:i + 2], FPI_wind[fpi]['North'][i:i + 2],
                        color=my_colors[fpi], label='_nolegend_', alpha=FPI_walpha[fpi]['North'][i + 1])
    axes01.scatter(FPI_ut[fpi]['North'], FPI_wind[fpi]['North'],
                   ec=None, color=my_colors[fpi], label='_nolegend_',
                   alpha=FPI_walpha[fpi]['North'], s=10)

    for i in range(len(FPI_ut[fpi]['West']) - 1):
        if (FPI_ut[fpi]['West'][i + 1] - FPI_ut[fpi]['West'][i]).total_seconds() < (max_gap * 60.):
            axes02.plot(FPI_ut[fpi]['West'][i:i + 2], FPI_wind[fpi]['West'][i:i + 2],
                        color=my_colors[fpi], label='_nolegend_', alpha=FPI_walpha[fpi]['West'][i + 1])
    axes02.scatter(FPI_ut[fpi]['West'], FPI_wind[fpi]['West'],
                   ec=None, color=my_colors[fpi], label='_nolegend_',
                   alpha=FPI_walpha[fpi]['West'], s=10)

    legend_elements.append(Line2D([0], [0], marker='o', linestyle='None',
                                  color=my_colors[fpi], label=fpi,
                                  markerfacecolor=my_colors[fpi], markersize=np.sqrt(10)))

axes01.set_ylim([params['ymin'], params['ymax']])
axes01.axhline(y=0, color='k', linewidth=1)
axes01.axvline(x=target_time, color='r', linewidth=1)
axes01.xaxis.set_major_locator(dates.HourLocator(interval=2))
axes01.xaxis.set_minor_locator(dates.HourLocator())
axes01.xaxis.set_major_formatter(dates.DateFormatter('%H'))
axes01.set_title(params['title'])
axes01.tick_params(axis='both', which='major', size=6, direction='in', right=True, top=True, labelright=True, labelleft=False)
axes01.tick_params(axis='both', which='minor', size=3, direction='in', right=True, top=True, labelright=True, labelleft=False)
axes01.set_ylabel('Northward Winds [m/s]')
axes01.legend(handles=legend_elements, loc='upper right', ncol=2, fontsize=8)
if len(all_fpi_times) > 0:
    axes01.set_xlim([min(all_fpi_times), max(all_fpi_times)])

axes02.set_ylim([params['ymin'], params['ymax']])
axes02.axhline(y=0, color='k', linewidth=1)
axes02.axvline(x=target_time, color='r', linewidth=1)
axes02.xaxis.set_major_locator(dates.HourLocator(interval=2))
axes02.xaxis.set_minor_locator(dates.HourLocator())
axes02.xaxis.set_major_formatter(dates.DateFormatter('%H'))
axes02.tick_params(axis='both', which='major', size=6, direction='in', right=True, top=True, labelright=True, labelleft=False)
axes02.tick_params(axis='both', which='minor', size=3, direction='in', right=True, top=True, labelright=True, labelleft=False)
axes02.yaxis.tick_right()
axes02.yaxis.set_label_position("left")
axes02.set_ylabel('Eastward Wind [m/s]')
axes02.set_xlabel('UT [hrs]')
if len(all_fpi_times) > 0:
    axes02.set_xlim([min(all_fpi_times), max(all_fpi_times)])

low_quality  = Line2D([], [], color=my_colors['cvo'], alpha=0.3, label='Low')
med_quality  = Line2D([], [], color=my_colors['cvo'], alpha=0.7, label='Medium')
high_quality = Line2D([], [], color=my_colors['cvo'], alpha=1.0, label='High Quality')
axes02.legend(handles=[low_quality, med_quality, high_quality], loc='upper right', ncols=3, fontsize=6)

width           = 0.015
headwidth       = 4
headlength      = 4
headaxislength  = headlength - 1
minshaft        = 2

# draw() is needed so axes00.bbox.width is populated before computing sc
fig.canvas.draw()
sc = axes00.bbox.width / fig.dpi / (10. * params['scale'])

obj = None
for fpi in FPI_ut.keys():
    valid_t = False

    if (np.min(abs(toTimestamp(pd.to_datetime(str(target_time))) - FPI_tt[fpi]['North'])) < max_fpi_gap * 60) and \
       (np.min(abs(toTimestamp(pd.to_datetime(str(target_time))) - FPI_tt[fpi]['West']))  < max_fpi_gap * 60):
        valid_t = True

    if valid_t:
        u = np.interp(toTimestamp(pd.to_datetime(str(target_time))), FPI_tt[fpi]['West'],  FPI_wind[fpi]['West'])  * sc
        v = np.interp(toTimestamp(pd.to_datetime(str(target_time))), FPI_tt[fpi]['North'], FPI_wind[fpi]['North']) * sc
        a = 1
    else:
        u = np.nan
        v = np.nan
        a = 0

    if a < 0.5:
        a = 0

    glat, glon, x = fpiinfo.get_site_info(fpi)['Location']

    obj = axes00.quiver(np.array([glon]), np.array([glat]), np.array([u]), np.array([v]),
                        angles='uv', scale_units='inches', scale=1, width=width,
                        pivot='tail', headwidth=headwidth, headlength=headlength, alpha=a,
                        minshaft=minshaft, headaxislength=headaxislength,
                        color=my_colors[fpi], transform=crs.PlateCarree(), zorder=1000)

x0, x1, y0, y1 = axes00.get_extent()
if obj is not None:
    quiverobj1 = axes00.quiverkey(obj, params['xoff'], params['yoff'], sc * params['scale'],
                                  r'$%i\,\frac{m}{s}$' % params['scale'],
                                  labelsep=0.05, color="black", alpha=1, coordinates='axes',
                                  labelpos='N', transform=axes00.transAxes,
                                  fontproperties={'size': 8})

today_date = pd.to_datetime('today').strftime('%m-%d-%Y')
if np.isnan(analysis_parameters['Tlo']) and np.isnan(analysis_parameters['Thi']):
    text1 = 'No filtering'
elif np.isnan(analysis_parameters['Tlo']):
    text1 = 'Highpass filter cutoff at %d min' % analysis_parameters['Thi']
elif np.isnan(analysis_parameters['Thi']):
    text1 = 'Lowpass filter cutoff at %d min' % analysis_parameters['Tlo']
else:
    text1 = 'Temporal filter: %d-%d min' % (analysis_parameters['Tlo'], analysis_parameters['Thi'])
text2 = 'Plotted on ' + today_date
text_x = 0
axes00.annotate(text1, xy=(text_x, -0.05),       xycoords='axes fraction', ha='left', va='bottom', fontsize=8)
axes00.annotate(text2, xy=(text_x, -0.05 - 0.05), xycoords='axes fraction', ha='left', va='bottom', fontsize=8)

with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='.*not compatible with tight_layout.*')
    spec.tight_layout(fig)

plt.show()

# ── Optional: save this frame ──────────────────────────────────────────────────
# savename = os.path.join(output_directory,
#     f"MANGO_{pd.to_datetime(str(target_time)).strftime('%Y%m%d_%H%M%S')}_{sky_line_tag}.png")
# plt.savefig(savename, dpi=150, bbox_inches='tight')
# logger.info(f"Saved: {savename}")


# Section 5 — Browse Available Times (Helper)

Use this cell to discover which times are available in the loaded night,
then copy the time string you want into `target_time_str` in Section 4 and re-run that cell.

In [ ]:
# Print all available frame times with their index
for i, t in enumerate(unique_times):
    print(f"[{i:4d}]  {pd.to_datetime(str(t)).strftime('%Y-%m-%d %H:%M:%S')} UT")


In [ ]:
import ipywidgets as widgets
from IPython.display import display

slider = widgets.IntSlider(
    value=0, min=0, max=len(unique_times) - 1, step=1,
    description='Frame index:', continuous_update=False,
    layout=widgets.Layout(width='600px')
)
label = widgets.Label()

def on_change(change):
    idx        = change['new']
    label.value = f"->  {pd.to_datetime(str(unique_times[idx])).strftime('%Y-%m-%d %H:%M:%S')} UT  (index {idx})"

slider.observe(on_change, names='value')
display(slider, label)
